# 02. Baseline RF & MLP Models

This notebook establishes reproducible clean baselines for Random Forest and MLP across seeds (42, 123, 2026), evaluates TPR/AUROC/AUPRC, and freezes the baseline models for subsequent robustness benchmarks.


MLP + RF BASELINE

In [22]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

In [23]:
# ============================================================
# REPRODUCIBLE CLEAN BASELINE: MLP + RANDOM FOREST
# UNSW-NB15 CLEAN DATASET
#
# Primary seeds:
#     42, 123, 2026
#
# Reports:
#     - Per-seed TPR
#     - Per-seed AUROC
#     - Per-seed AUPRC
#     - Mean ± SD across the three seeds
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import random
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEEDS = [42, 123, 2026]


def set_seed(seed):
    """
    Set all relevant random seeds for reproducible experiments.
    """

    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch CPU
    torch.manual_seed(seed)

    # PyTorch CUDA
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic CUDA behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Where supported by the installed PyTorch version
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


# ============================================================
# 2. DATA
# ============================================================

TARGET = "label"

# attack_cat directly describes the attack category.
# It is therefore excluded to prevent label leakage.
DROP_COLS = [
    "label",
    "attack_cat"
]


# ------------------------------------------------------------
# IMPORTANT:
#
# train_df and test_df are assumed to have already been
# created using the FINAL dataset split.
#
# This cell does NOT recreate the train/test split.
# ------------------------------------------------------------

feature_cols = [
    c for c in train_df.columns
    if c not in DROP_COLS
]


X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df[TARGET].astype(int).to_numpy()
y_test = test_df[TARGET].astype(int).to_numpy()


print("=" * 80)
print("CLEAN BASELINE DATA")
print("=" * 80)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain labels:")
print(pd.Series(y_train).value_counts())

print("\nTest labels:")
print(pd.Series(y_test).value_counts())


# ============================================================
# 3. IDENTIFY FEATURE TYPES
# ============================================================

categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_cols = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()


print("\nNumeric features    :", len(numeric_cols))
print("Categorical features:", len(categorical_cols))

print("\nCategorical columns:")
print(categorical_cols)


# ============================================================
# 4. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])


preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numeric_cols
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_cols
    )
])


print("\n" + "=" * 80)
print("PREPROCESSING")
print("=" * 80)

print("Fitting preprocessor on training data...")


# IMPORTANT:
# Fit ONLY on training data.
X_train_processed = preprocessor.fit_transform(
    X_train
)

# Test data is transformed using the SAME fitted preprocessor.
X_test_processed = preprocessor.transform(
    X_test
)


print(
    "Processed train shape:",
    X_train_processed.shape
)

print(
    "Processed test shape :",
    X_test_processed.shape
)


# ============================================================
# 5. CONVERT DATA TO NUMPY
# ============================================================

X_train_np = np.asarray(
    X_train_processed,
    dtype=np.float32
)

X_test_np = np.asarray(
    X_test_processed,
    dtype=np.float32
)

y_train_np = y_train.astype(
    np.float32
)

y_test_np = y_test.astype(
    np.float32
)


# ============================================================
# 6. MLP ARCHITECTURE
# ============================================================

class MLP(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.Linear(
                32,
                1
            )
        )

    def forward(self, x):

        return self.network(
            x
        ).squeeze(1)


# ============================================================
# 7. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)


# ============================================================
# 8. FUNCTION: TRAIN + EVALUATE MLP
# ============================================================

def run_mlp(seed):

    print("\n")
    print("=" * 80)
    print(f"MLP — SEED {seed}")
    print("=" * 80)

    # --------------------------------------------------------
    # Set seed BEFORE creating model and DataLoader
    # --------------------------------------------------------

    set_seed(seed)


    # --------------------------------------------------------
    # PyTorch tensors
    # --------------------------------------------------------

    X_train_tensor = torch.tensor(
        X_train_np,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train_np,
        dtype=torch.float32
    )

    X_test_tensor = torch.tensor(
        X_test_np,
        dtype=torch.float32
    )


    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )


    # --------------------------------------------------------
    # Deterministic DataLoader
    # --------------------------------------------------------

    generator = torch.Generator()

    generator.manual_seed(seed)


    train_loader = DataLoader(
        train_dataset,
        batch_size=512,
        shuffle=True,
        generator=generator,
        num_workers=0
    )


    # --------------------------------------------------------
    # Create model AFTER setting seed
    # --------------------------------------------------------

    model = MLP(
        X_train_np.shape[1]
    ).to(device)


    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    EPOCHS = 10

    print("\nTraining MLP...")

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0.0


        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(device)

            batch_y = batch_y.to(device)


            optimizer.zero_grad()


            logits = model(
                batch_X
            )


            loss = criterion(
                logits,
                batch_y
            )


            loss.backward()


            optimizer.step()


            total_loss += (
                loss.item()
                * len(batch_X)
            )


        avg_loss = (
            total_loss
            / len(train_dataset)
        )


        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} "
            f"- Loss: {avg_loss:.6f}"
        )


    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    model.eval()


    with torch.no_grad():

        mlp_logits = model(
            X_test_tensor.to(device)
        )

        mlp_prob = torch.sigmoid(
            mlp_logits
        ).cpu().numpy()


    # --------------------------------------------------------
    # Classification at threshold = 0.5
    # --------------------------------------------------------

    mlp_pred = (
        mlp_prob >= 0.5
    ).astype(int)


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    mlp_tpr = recall_score(
        y_test,
        mlp_pred,
        zero_division=0
    )

    mlp_auroc = roc_auc_score(
        y_test,
        mlp_prob
    )

    mlp_auprc = average_precision_score(
        y_test,
        mlp_prob
    )


    print("\nMLP RESULTS")
    print("-" * 40)

    print(
        f"Seed  : {seed}"
    )

    print(
        f"TPR   : {mlp_tpr:.6f}"
    )

    print(
        f"AUROC : {mlp_auroc:.6f}"
    )

    print(
        f"AUPRC : {mlp_auprc:.6f}"
    )


    return {
        "Seed": seed,
        "TPR": mlp_tpr,
        "AUROC": mlp_auroc,
        "AUPRC": mlp_auprc
    }


# ============================================================
# 9. FUNCTION: TRAIN + EVALUATE RANDOM FOREST
# ============================================================

def run_random_forest(seed):

    print("\n")
    print("=" * 80)
    print(f"RANDOM FOREST — SEED {seed}")
    print("=" * 80)


    # Set NumPy/Python randomness for reproducibility
    set_seed(seed)


    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------

    rf = RandomForestClassifier(
        n_estimators=200,
        random_state=seed,
        n_jobs=-1
    )


    print("Training Random Forest...")


    rf.fit(
        X_train_processed,
        y_train
    )


    # --------------------------------------------------------
    # Probability predictions
    # --------------------------------------------------------

    rf_prob = rf.predict_proba(
        X_test_processed
    )[:, 1]


    # --------------------------------------------------------
    # Classification at threshold = 0.5
    # --------------------------------------------------------

    rf_pred = (
        rf_prob >= 0.5
    ).astype(int)


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    rf_tpr = recall_score(
        y_test,
        rf_pred,
        zero_division=0
    )

    rf_auroc = roc_auc_score(
        y_test,
        rf_prob
    )

    rf_auprc = average_precision_score(
        y_test,
        rf_prob
    )


    print("\nRF RESULTS")
    print("-" * 40)

    print(
        f"Seed  : {seed}"
    )

    print(
        f"TPR   : {rf_tpr:.6f}"
    )

    print(
        f"AUROC : {rf_auroc:.6f}"
    )

    print(
        f"AUPRC : {rf_auprc:.6f}"
    )


    return {
        "Seed": seed,
        "TPR": rf_tpr,
        "AUROC": rf_auroc,
        "AUPRC": rf_auprc
    }


# ============================================================
# 10. RUN MLP ACROSS ALL PRIMARY SEEDS
# ============================================================

mlp_results = []


for seed in SEEDS:

    result = run_mlp(seed)

    mlp_results.append(
        result
    )


mlp_results_df = pd.DataFrame(
    mlp_results
)


# ============================================================
# 11. RUN RF ACROSS ALL PRIMARY SEEDS
# ============================================================

rf_results = []


for seed in SEEDS:

    result = run_random_forest(seed)

    rf_results.append(
        result
    )


rf_results_df = pd.DataFrame(
    rf_results
)


# ============================================================
# 12. CALCULATE MEAN ± SD
# ============================================================

mlp_mean = mlp_results_df[
    ["TPR", "AUROC", "AUPRC"]
].mean()

mlp_std = mlp_results_df[
    ["TPR", "AUROC", "AUPRC"]
].std(
    ddof=1
)


rf_mean = rf_results_df[
    ["TPR", "AUROC", "AUPRC"]
].mean()

rf_std = rf_results_df[
    ["TPR", "AUROC", "AUPRC"]
].std(
    ddof=1
)


# ============================================================
# 13. FINAL CLEAN BASELINE TABLE
# ============================================================

final_baseline = pd.DataFrame({

    "Model": [
        "Random Forest",
        "MLP"
    ],

    "TPR": [
        f"{rf_mean['TPR']:.4f} ± {rf_std['TPR']:.4f}",
        f"{mlp_mean['TPR']:.4f} ± {mlp_std['TPR']:.4f}"
    ],

    "AUROC": [
        f"{rf_mean['AUROC']:.4f} ± {rf_std['AUROC']:.4f}",
        f"{mlp_mean['AUROC']:.4f} ± {mlp_std['AUROC']:.4f}"
    ],

    "AUPRC": [
        f"{rf_mean['AUPRC']:.4f} ± {rf_std['AUPRC']:.4f}",
        f"{mlp_mean['AUPRC']:.4f} ± {mlp_std['AUPRC']:.4f}"
    ]
})


# ============================================================
# 14. PRINT PER-SEED RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("MLP RESULTS ACROSS SEEDS")
print("=" * 80)

print(
    mlp_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\n")
print("=" * 80)
print("RANDOM FOREST RESULTS ACROSS SEEDS")
print("=" * 80)

print(
    rf_results_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 15. FINAL BASELINE
# ============================================================

print("\n")
print("=" * 80)
print("FINAL CLEAN BASELINE — MEAN ± SD")
print("=" * 80)

print(
    final_baseline.to_string(
        index=False
    )
)


# ============================================================
# 16. SAVE RESULTS
# ============================================================

mlp_results_df.to_csv(
    "MLP_clean_baseline_by_seed.csv",
    index=False
)

rf_results_df.to_csv(
    "RF_clean_baseline_by_seed.csv",
    index=False
)

final_baseline.to_csv(
    "clean_baseline_mean_sd.csv",
    index=False
)


print("\n")
print("=" * 80)
print("RESULT FILES SAVED")
print("=" * 80)

print("MLP_clean_baseline_by_seed.csv")
print("RF_clean_baseline_by_seed.csv")
print("clean_baseline_mean_sd.csv")

CLEAN BASELINE DATA
X_train: (175341, 43)
X_test : (82332, 43)

Train labels:
1    119341
0     56000
Name: count, dtype: int64

Test labels:
1    45332
0    37000
Name: count, dtype: int64

Numeric features    : 40
Categorical features: 3

Categorical columns:
['proto', 'service', 'state']

PREPROCESSING
Fitting preprocessor on training data...
Processed train shape: (175341, 195)
Processed test shape : (82332, 195)

Device: cuda


MLP — SEED 42

Training MLP...
Epoch 01/10 - Loss: 0.160476
Epoch 02/10 - Loss: 0.099631
Epoch 03/10 - Loss: 0.091877
Epoch 04/10 - Loss: 0.083007
Epoch 05/10 - Loss: 0.072669
Epoch 06/10 - Loss: 0.064407
Epoch 07/10 - Loss: 0.060630
Epoch 08/10 - Loss: 0.056684
Epoch 09/10 - Loss: 0.054349
Epoch 10/10 - Loss: 0.052174

MLP RESULTS
----------------------------------------
Seed  : 42
TPR   : 0.437483
AUROC : 0.671884
AUPRC : 0.753734


MLP — SEED 123

Training MLP...
Epoch 01/10 - Loss: 0.161831
Epoch 02/10 - Loss: 0.097787
Epoch 03/10 - Loss: 0.085886
Epoch

In [24]:
# ============================================================
# FREEZE THE EXACT CLEAN BASELINE MODELS
#
# 3 MLPs + 3 Random Forests
# Seeds: 42, 123, 2026
#
# These SAME models will be used for ALL robustness tests.
# NO retraining during robustness experiments.
# ============================================================

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)


print("=" * 80)
print("FREEZING CLEAN BASELINE MODELS")
print("=" * 80)


# ============================================================
# 1. SEEDS
# ============================================================

SEEDS = [42, 123, 2026]


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


# ============================================================
# 3. CHECK DATA
# ============================================================

assert "X_train_np" in globals()
assert "X_test_np" in globals()
assert "y_train_np" in globals()
assert "y_test" in globals()
assert "preprocessor" in globals()

print("Training data :", X_train_np.shape)
print("Testing data  :", X_test_np.shape)


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# 5. MLP ARCHITECTURE
# EXACT SAME ARCHITECTURE AS BASELINE
# ============================================================

class MLP(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):

        return self.network(x).squeeze(1)


# ============================================================
# 6. STORAGE
# ============================================================

# THESE ARE THE IMPORTANT VARIABLES
# They contain the actual frozen models.

mlp_models = {}
rf_models = {}

mlp_baseline_results = []
rf_baseline_results = []


# ============================================================
# 7. TRAIN MLPs
# ============================================================

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"TRAINING + FREEZING MLP — SEED {seed}")
    print("=" * 80)

    set_seed(seed)

    # --------------------------------------------------------
    # tensors
    # --------------------------------------------------------

    X_train_tensor = torch.tensor(
        X_train_np,
        dtype=torch.float32
    )

    y_train_tensor = torch.tensor(
        y_train_np,
        dtype=torch.float32
    )

    X_test_tensor = torch.tensor(
        X_test_np,
        dtype=torch.float32
    )


    # --------------------------------------------------------
    # dataset
    # --------------------------------------------------------

    train_dataset = TensorDataset(
        X_train_tensor,
        y_train_tensor
    )


    # --------------------------------------------------------
    # deterministic DataLoader
    # --------------------------------------------------------

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=512,
        shuffle=True,
        generator=generator,
        num_workers=0
    )


    # --------------------------------------------------------
    # CREATE MODEL
    # --------------------------------------------------------

    mlp = MLP(
        X_train_np.shape[1]
    ).to(device)


    # --------------------------------------------------------
    # LOSS
    # --------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()


    # --------------------------------------------------------
    # OPTIMIZER
    # --------------------------------------------------------

    optimizer = torch.optim.Adam(
        mlp.parameters(),
        lr=1e-3
    )


    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    EPOCHS = 10

    for epoch in range(EPOCHS):

        mlp.train()

        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = mlp(batch_X)

            loss = criterion(
                logits,
                batch_y
            )

            loss.backward()

            optimizer.step()


    # --------------------------------------------------------
    # EVALUATE CLEAN TEST SET
    # --------------------------------------------------------

    mlp.eval()

    with torch.no_grad():

        logits = mlp(
            X_test_tensor.to(device)
        )

        prob = torch.sigmoid(
            logits
        ).cpu().numpy()


    pred = (
        prob >= 0.5
    ).astype(int)


    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    tpr = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    auroc = roc_auc_score(
        y_test,
        prob
    )

    auprc = average_precision_score(
        y_test,
        prob
    )


    # --------------------------------------------------------
    # FREEZE MODEL
    # --------------------------------------------------------

    mlp.eval()

    for parameter in mlp.parameters():

        parameter.requires_grad = False


    # STORE THE ACTUAL MODEL
    mlp_models[seed] = mlp


    mlp_baseline_results.append({

        "Seed": seed,
        "TPR": tpr,
        "AUROC": auroc,
        "AUPRC": auprc
    })


    print(
        f"Seed {seed}: "
        f"TPR={tpr:.6f}, "
        f"AUROC={auroc:.6f}, "
        f"AUPRC={auprc:.6f}"
    )


# ============================================================
# 8. TRAIN RANDOM FORESTS
# ============================================================

for seed in SEEDS:

    print("\n" + "=" * 80)
    print(f"TRAINING + FREEZING RF — SEED {seed}")
    print("=" * 80)

    set_seed(seed)


    rf_model = RandomForestClassifier(
        n_estimators=200,
        random_state=seed,
        n_jobs=-1
    )


    rf_model.fit(
        X_train_processed,
        y_train
    )


    # --------------------------------------------------------
    # CLEAN TEST
    # --------------------------------------------------------

    prob = rf_model.predict_proba(
        X_test_processed
    )[:, 1]


    pred = (
        prob >= 0.5
    ).astype(int)


    tpr = recall_score(
        y_test,
        pred,
        zero_division=0
    )

    auroc = roc_auc_score(
        y_test,
        prob
    )

    auprc = average_precision_score(
        y_test,
        prob
    )


    # --------------------------------------------------------
    # STORE ACTUAL MODEL
    # --------------------------------------------------------

    rf_models[seed] = rf_model


    rf_baseline_results.append({

        "Seed": seed,
        "TPR": tpr,
        "AUROC": auroc,
        "AUPRC": auprc
    })


    print(
        f"Seed {seed}: "
        f"TPR={tpr:.6f}, "
        f"AUROC={auroc:.6f}, "
        f"AUPRC={auprc:.6f}"
    )


# ============================================================
# 9. BASELINE SUMMARY
# ============================================================

mlp_baseline_df = pd.DataFrame(
    mlp_baseline_results
)

rf_baseline_df = pd.DataFrame(
    rf_baseline_results
)


print("\n")
print("=" * 80)
print("FROZEN CLEAN BASELINE")
print("=" * 80)


print("\nMLP")
print(
    mlp_baseline_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nMLP MEAN:")
print(
    mlp_baseline_df[
        ["TPR", "AUROC", "AUPRC"]
    ].mean()
)

print("\nMLP SD:")
print(
    mlp_baseline_df[
        ["TPR", "AUROC", "AUPRC"]
    ].std(ddof=1)
)


print("\nRANDOM FOREST")
print(
    rf_baseline_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nRF MEAN:")
print(
    rf_baseline_df[
        ["TPR", "AUROC", "AUPRC"]
    ].mean()
)

print("\nRF SD:")
print(
    rf_baseline_df[
        ["TPR", "AUROC", "AUPRC"]
    ].std(ddof=1)
)


# ============================================================
# 10. CONFIRM MODELS ARE FROZEN
# ============================================================

print("\n")
print("=" * 80)
print("MODEL FREEZE CHECK")
print("=" * 80)

print(
    "Stored MLP seeds:",
    list(mlp_models.keys())
)

print(
    "Stored RF seeds:",
    list(rf_models.keys())
)

assert set(mlp_models.keys()) == set(SEEDS)
assert set(rf_models.keys()) == set(SEEDS)

print("\nSUCCESS:")
print("The same 3 MLPs and 3 RFs are now frozen.")
print("Use ONLY mlp_models and rf_models for robustness experiments.")

FREEZING CLEAN BASELINE MODELS
Training data : (175341, 195)
Testing data  : (82332, 195)
Device: cuda

TRAINING + FREEZING MLP — SEED 42
Seed 42: TPR=0.437483, AUROC=0.671884, AUPRC=0.753734

TRAINING + FREEZING MLP — SEED 123
Seed 123: TPR=0.403909, AUROC=0.665196, AUPRC=0.748217

TRAINING + FREEZING MLP — SEED 2026
Seed 2026: TPR=0.429167, AUROC=0.647277, AUPRC=0.746170

TRAINING + FREEZING RF — SEED 42
Seed 42: TPR=0.716712, AUROC=0.805111, AUPRC=0.819160

TRAINING + FREEZING RF — SEED 123
Seed 123: TPR=0.656755, AUROC=0.796718, AUPRC=0.813893

TRAINING + FREEZING RF — SEED 2026
Seed 2026: TPR=0.675373, AUROC=0.796694, AUPRC=0.812162


FROZEN CLEAN BASELINE

MLP
 Seed      TPR    AUROC    AUPRC
   42 0.437483 0.671884 0.753734
  123 0.403909 0.665196 0.748217
 2026 0.429167 0.647277 0.746170

MLP MEAN:
TPR      0.423520
AUROC    0.661452
AUPRC    0.749374
dtype: float64

MLP SD:
TPR      0.017485
AUROC    0.012723
AUPRC    0.003912
dtype: float64

RANDOM FOREST
 Seed      TPR    AU